# SNI-21 — frozen A0 per source domain

Evaluasi checkpoint A0 yang sama pada validation Adrian dan Faruq. Notebook ini tidak training dan tidak memulihkan test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import coffee_detector
print('IMPORT:', coffee_detector.__file__)


In [ ]:
DRIVE_ROOTS = [Path('/content/drive/MyDrive'), Path('/content/drive/.shortcut-targets-by-id')]

def find_artifact(name, preferred=()):
    for path in preferred:
        if path.is_file():
            return path
    matches = []
    for root in DRIVE_ROOTS:
        if root.is_dir():
            matches.extend(path for path in root.rglob(name) if path.is_file())
    matches = sorted(set(matches))
    assert matches, f'{name} tidak ditemukan di MyDrive/shortcut.'
    if len(matches) > 1:
        print(f'{name}: {len(matches)} kandidat; memakai {matches[0]}')
    return matches[0]

A0_ARCHIVE = find_artifact('A0_real.tar', [
    Path('/content/drive/MyDrive/Coffee_Bean_Detection/bundles/sni21-vadcp-pilot-bundle/A0_real.tar'),
])
checkpoint_preferred = Path('/content/drive/MyDrive/Coffee_Bean_Detection/checkpoints/sni21-vadcp-pilot-results/A0_seed42/weights/best.pt')
if checkpoint_preferred.is_file():
    CHECKPOINT = checkpoint_preferred
else:
    candidates = []
    for root in DRIVE_ROOTS:
        if root.is_dir():
            candidates.extend(root.rglob('A0_seed42/weights/best.pt'))
    candidates = sorted(path for path in set(candidates) if path.is_file())
    assert candidates, 'Checkpoint khusus A0_seed42 tidak ditemukan.'
    CHECKPOINT = candidates[0]

COMBINED_ROOT = Path('/content/sni21-a0-development')
SEPARATED_ROOT = Path('/content/sni21-source-separated-v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments/sni21-source-domain-evaluation-v1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ARCHIVE   :', A0_ARCHIVE)
print('CHECKPOINT:', CHECKPOINT)
print('OUTPUT    :', OUTPUT_ROOT)


In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_development
from coffee_detector.separate_sni21_sources import separate_sni21_sources

restore_real_a0_development(A0_ARCHIVE, COMBINED_ROOT)
assert not (COMBINED_ROOT / 'test').exists()
separation = separate_sni21_sources(COMBINED_ROOT, SEPARATED_ROOT, link_mode='auto')
assert separation['training_executed'] is False
assert separation['test_images_accessed'] is False
print('PEMISAHAN SIAP')


In [ ]:
import json
from coffee_detector.evaluate_sni21_source_domains import evaluate_sni21_source_domains

summary = evaluate_sni21_source_domains(
    CHECKPOINT, SEPARATED_ROOT, OUTPUT_ROOT, device='0'
)
assert summary['training_executed'] is False
assert summary['test_images_accessed'] is False
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
import pandas as pd
from IPython.display import display

table = pd.DataFrame(summary['rows'])
percent = ['map50_95', 'map50', 'precision', 'recall', 'macro_map50_95', 'bottom3_map50_95', 'worst_map50_95']
display(table.style.format({name: '{:.2%}' for name in percent}))
print('TRAINING:', summary['training_executed'])
print('TEST ACCESSED:', summary['test_images_accessed'])
print('SUMMARY:', summary['summary'])
print('Kirim tabel ini. Jangan training model baru.')
